## Pipeline Drafts - Early Iterations (v1.0)

First attempt at building the pipeline. Many issues with geo matching.

In [ ]:
# Attempt 1: Load raw data
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Load first source
df1 = pd.read_csv(RAW_DIR / "tunisia-real-estate.csv")
print(f"Loaded {len(df1)} rows from tunisia-real-estate")

Issue: The locality names don't match our delegation list!

In [ ]:
# Try fuzzy matching - this was slow and inaccurate
from difflib import get_close_matches

def match_location_fuzzy(name, candidates, n=1):
    return get_close_matches(name, candidates, n=n, cutoff=0.6)

# This took forever and gave wrong matches
sample_locations = df1['Locality'].dropna().unique()[:10]
print("Sample locations:", sample_locations)

## Iteration 2 (v1.2) - More geo fixes

In [ ]:
# Second attempt: Try adding more data sources
df2 = pd.read_csv(RAW_DIR / "Property Prices in Tunisia.csv")
df3 = pd.read_csv(RAW_DIR / "data_prices_cleaned.csv")

print(f"Source 2: {len(df2)} rows")
print(f"Source 3: {len(df3)} rows")

In [ ]:
# Concatenate all sources
df_all = pd.concat([df1, df2, df3], ignore_index=True)
print(f"Combined: {len(df_all)} rows")

# Problem: Still no governorate column in some!

## Iteration 3 (v1.5) - Debugging the merge

In [ ]:
# Check columns
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("df3 columns:", df3.columns.tolist())

In [ ]:
# Map columns manually - lots of trial and error
column_mapping = {
    'Price': 'price_tnd',
    'price': 'price_tnd',
    'Surface': 'surface_m2',
    'size': 'surface_m2',
    'superficie': 'surface_m2'
}

# This was tedious...

## Iteration 4 (v1.8) - Model Training First Try

In [ ]:
# First model attempt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Simple features
X = df_all[['surface_m2', 'rooms']].fillna(0)
y = df_all['price_tnd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

rf = RandomForestRegressor(n_estimators=50)
rf.fit(X_train, y_train)

score = rf.score(X_test, y_test)
print(f"First model R²: {score:.3f}")

# Very poor! Only 0.45 or so

## Iteration 5 (v1.9) - Debugging features

In [ ]:
# Add geographic features
# Problem: How to encode location?
from sklearn.preprocessing import LabelEncoder

le_deleg = LabelEncoder()
df_all['geo_delegation_enc'] = le_deleg.fit_transform(df_all['geo_delegation'].fillna('unknown'))

# Still weak - need better encoding

## Iteration 6 (v2.0) - Target Encoding Attempt

In [ ]:
# Try target encoding - mean price per delegation
delegation_means = df_all.groupby('geo_delegation')['price_tnd'].mean()
df_all['geo_delegation_target_enc'] = df_all['geo_delegation'].map(delegation_means)

# Better! R² went up to 0.72

## Iteration 7 (v2.1) - Coordinate Features

In [ ]:
# Add lat/lon from external geo data
# This is where we loaded the geo/ folder
geo_coords = pd.read_csv("../geo/delegations_coordinates.csv")
print(f"Loaded {len(geo_coords)} delegation coordinates")

# Merge on delegation name
df_all = df_all.merge(geo_coords, on='geo_delegation', how='left')
print(f"After merge: {df_all['lat'].notna().sum()} / {len(df_all)} have coords")

## Iteration 8 (v2.2) - GradientBoosting Switch

In [ ]:
# Switch from RandomForest to GradientBoosting
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5)
gb.fit(X_train, y_train)

gb_score = gb.score(X_test, y_test)
print(f"GradientBoosting R²: {gb_score:.3f}")

# Much better: 0.89!

## Iteration 9 (v2.3) - Coverage Issues

In [ ]:
# Check coverage - many delegations have no data
delegation_counts = df_all.groupby('geo_delegation').size()
low_support = (delegation_counts < 10).sum()
print(f"delegations with < 10 rows: {low_support}")

# Need fallback system!

## Iteration 10 (v2.4) - Fallback Implementation

In [ ]:
# Hierarchy fallback: delegation -> governorate -> national
def get_coverage_level(delegation_count):
    if delegation_count >= 20:
        return 'direct'
    elif delegation_count >= 5:
        return 'locality'
    return 'fallback'

df_all['coverage_level'] = df_all.groupby('geo_delegation')['price_tnd'].transform('count').apply(get_coverage_level)
print(df_all['coverage_level'].value_counts())

## Final Notes

This was the working version before we finalized the pipeline structure.

In [ ]:
# Export for next stage
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

df_all.to_csv(PROCESSED_DIR / "02_cleaning" / "02_pipeline_v2.4.csv", index=False)
print("Saved draft output")